# Initial Setup

1. Construct a StateGraph
2. Add LLM Processing Node
3. Add Edges
4. Compile Graph for execution

In [0]:
from dotenv import load_dotenv
import os
import random
from typing import Literal, TypedDict
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated
from databricks_langchain import ChatDatabricks
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import MessagesState

Step 1: Construct a StateGraph

In [0]:
builder = StateGraph(MessagesState)

Step 2: Add LLM as proessing node

MessageState - holds the state of all messages passed through the graph (between the nodes)

In [0]:
llm = ChatDatabricks(endpoint = "databricks-gpt-oss-120b")

def call_llm(state: MessagesState):

    system_msg = {"role": "system", "content": "Reply only with plain text. No formatting."}
    all_msgs = [system_msg] + state["messages"]

    return {"messages": [llm.invoke(all_msgs)]}

In [0]:
builder.add_node("call_llm", call_llm)

Step 3: Add Edges

In [0]:
builder.add_edge(START, "call_llm")
builder.add_edge("call_llm", END)

Step 4: Compile Graph for execution

In [0]:
graph = builder.compile()

Step 5: Execute and print the results

In [0]:
messages = graph.invoke({"messages": [HumanMessage("Tell me more about langgraph ?")]})

In [0]:
message = messages['messages'][-1]

if isinstance(message.content, str):
    ai_message = message.content
else:
    for part in message.content:
        if part.get("type") == "text":
           ai_message = part.get("text")
           break

print(ai_message)